In [5]:
from preprocessing.data_cleaning import preprocess
from config import DTYPE_MAPPING

In [6]:
from pathlib import Path
DATA_DIR = Path("../data").resolve() / "melb_jun_25"                 
LISTINGS_CSV = DATA_DIR / "listings.csv"
REVIEWS_CSV  = DATA_DIR / "reviews.csv"
OUTPUT_DIR = Path("../output").resolve()

In [7]:
import pandas as pd
df = pd.read_csv(LISTINGS_CSV)

In [8]:
processed_df, imputation_pipeline = preprocess(df, DTYPE_MAPPING)

In [5]:
len(processed_df)

18539

In [25]:
processed_df["bathrooms_text"].value_counts()

bathrooms_text
1 bath               8833
2 baths              3859
1 shared bath        1335
1 private bath       1275
1.5 baths             780
2.5 baths             681
3 baths               400
1.5 shared baths      283
2 shared baths        202
3.5 baths             193
2.5 shared baths      119
4 baths                85
Shared half-bath       79
0 baths                57
0 shared baths         54
Half-bath              46
4.5 baths              39
Private half-bath      36
3 shared baths         31
5 baths                21
3.5 shared baths       15
8 baths                15
8 shared baths         12
5.5 baths              12
7.5 shared baths       10
6 baths                 7
7.5 baths               5
5.5 shared baths        5
9 baths                 4
6.5 baths               4
4 shared baths          4
7 baths                 4
7 shared baths          3
9.5 baths               2
6 shared baths          2
8.5 shared baths        2
5 shared baths          2
8.5 baths              

In [10]:
import pandas as pd
import numpy as np

def preprocess_datetime(df, date_cols, reference_date=None, fill_na=True):
    df = df.copy()
    if reference_date is None:
        reference_date = pd.Timestamp.today()
    else:
        reference_date = pd.to_datetime(reference_date)
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        if fill_na:
            median_date = df[col].median()
            df[col] = df[col].fillna(median_date)
        df[f"{col}_year"] = df[col].dt.year
        df[f"{col}_month"] = df[col].dt.month
        df[f"{col}_day"] = df[col].dt.day
        df[f"{col}_dayofweek"] = df[col].dt.dayofweek
        df[f"{col}_is_weekend"] = df[col].dt.dayofweek.isin([5,6]).astype(int)
        df[f"{col}_quarter"] = df[col].dt.quarter
        df[f"{col}_month_sin"] = np.sin(2 * np.pi * df[f"{col}_month"] / 12)
        df[f"{col}_month_cos"] = np.cos(2 * np.pi * df[f"{col}_month"] / 12)
        df[f"{col}_dow_sin"] = np.sin(2 * np.pi * df[f"{col}_dayofweek"] / 7)
        df[f"{col}_dow_cos"] = np.cos(2 * np.pi * df[f"{col}_dayofweek"] / 7)
        df[f"{col}_days_since"] = (reference_date - df[col]).dt.days
    return df

In [11]:
processed_df = preprocess_datetime(processed_df, date_cols=DTYPE_MAPPING['date'])

In [12]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re

def categorical_encoding_pipeline(mapping):
    return (
        ColumnTransformer(
            [
                ("onehot", OneHotEncoder(sparse_output=False), mapping["nominal"]),
                #("multihot", CountVectorizer(binary=True), ["amenities", "host_verifications"])
            ],
            remainder='passthrough'
        ),
        mapping["nominal"]
    )

def categorical_encoding_fit_transform(preprocessor, df, target_cols):
    result_array = preprocessor.fit_transform(df)
    
    columns = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "onehot":
            columns.extend(preprocessor.named_transformers_['onehot'].get_feature_names_out(cols))
        elif name == "remainder":
            passthrough_cols = [c for c in df.columns if c not in target_cols]
            columns.extend(passthrough_cols)
    
    return pd.DataFrame(result_array, columns=columns, index=df.index)

In [13]:
preprocessor, target_cols = categorical_encoding_pipeline(DTYPE_MAPPING)
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('onehot', OneHotEncoder(sparse_output=False),
                                 ['host_response_time', 'room_type'])])

In [14]:
processed_df = categorical_encoding_fit_transform(preprocessor, processed_df, target_cols)

In [15]:
def clean_boolean(entry):
    if (entry == 't'):
        return 1
    else:
        return 0

for col in DTYPE_MAPPING["boolean"]:
    processed_df[col] = processed_df[col].apply(clean_boolean)

In [16]:
processed_df[DTYPE_MAPPING["boolean"]]

,host_is_superhost,host_has_profile_pic,host_identity_verified,has_availability,instant_bookable
0,1,1,1,1,0
3,1,1,1,1,0
4,1,1,1,1,0
5,0,1,1,1,0
6,1,1,1,1,0
...,...,...,...,...,...
25796,0,1,1,1,0
25797,0,1,1,1,1
25798,0,1,1,1,0
25799,0,1,1,1,0


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
import pandas as pd

# preprocess property_type
def text_multihot(df, target_col):
    pipe = Pipeline([
        ('to_1d', FunctionTransformer(lambda x: x[target_col].astype(str).values, validate=False)),
        ('cv', CountVectorizer(binary=True, lowercase=True, stop_words='english', ngram_range=(1,2)))
    ])

    X = pipe.fit_transform(df)
    cv = pipe.named_steps['cv']
    col_names = [f"{target_col}_{feat}" for feat in cv.get_feature_names_out()]
    encoded = pd.DataFrame(X.toarray(), columns=col_names, index=df.index)
    processed_df = pd.concat([df.drop(columns=[target_col]), encoded], axis=1)
    return processed_df

processed_df = text_multihot(processed_df, "property_type")


Index(['host_response_time_a few days or more',
       'host_response_time_response time not mentioned',
       'host_response_time_within a day',
       'host_response_time_within a few hours',
       'host_response_time_within an hour', 'room_type_Entire home/apt',
       'room_type_Hotel room', 'room_type_Private room',
       'room_type_Shared room', 'latitude',
       ...
       'property_type_tiny', 'property_type_tiny home',
       'property_type_townhouse', 'property_type_train',
       'property_type_treehouse', 'property_type_unit',
       'property_type_vacation', 'property_type_vacation home',
       'property_type_villa', 'property_type_yurt'],
      dtype='object', length=247)


In [28]:
import re
import pandas as pd

def split_bathroom_info(df, col):
    num = df[col].str.extract(r'(\d+(?:\.\d+)?)').astype(float)
    

    num = num[0].fillna(
        df[col].str.contains('half', case=False, na=False).map({True: 0.5, False: 1})
    )

    df['bathroom_number'] = num
    df['bathroom_private'] = df[col].str.contains('private', case=False, na=False).astype(int)
    df['bathroom_shared'] = df[col].str.contains('shared', case=False, na=False).astype(int)
    
    return df.drop(columns=[col])

processed_df = split_bathroom_info(processed_df, 'bathrooms_text')



In [30]:
print(processed_df[['bathroom_number', 'bathroom_private', 'bathroom_shared']].tail())

       bathroom_number  bathroom_private  bathroom_shared
25796              2.0                 0                0
25797              1.0                 0                0
25798              1.0                 1                0
25799              1.0                 1                0
25800              1.0                 0                0


In [13]:
def get_percent_nan(df, target):
    return df[target].isnull().sum() / len(df)

In [14]:
processed_df['amenities']

0        [Room-darkening shades, Kitchen, BBQ grill: ga...
3        [Kitchen, Stainless steel single oven, Private...
4        [Room-darkening shades, Kitchen, Fire extingui...
5        [AC - split type ductless system, Kitchen, Sto...
6        [AC - split type ductless system, Kitchen, Win...
                               ...                        
25796    [Kitchen, Dedicated workspace, Beach access – ...
25797    [AC - split type ductless system, Kitchen, Pri...
25798    [Kitchen, Dedicated workspace, Lock on bedroom...
25799    [Kitchen, Dedicated workspace, Lake access, Wa...
25800    [Room-darkening shades, Kitchen, Stove, Clothi...
Name: amenities, Length: 18539, dtype: object

In [77]:
for c in processed_df.columns:
    if (n := get_percent_nan(processed_df,c)) > 0:
        print(c)
        print(n)

description
0.013161443443551432
neighborhood_overview
0.547710232482874
host_about
0.4287717784130751
